# Codigo 

In [20]:
import cv2
import numpy as np
import pandas as pd
import math

# Variables para almacenar los puntos del círculo y las regiones
circulo_puntos = []
valores_regiones = []
dibujando_circulo = False
regiones_dibujadas = False

# Función para definir y asignar valores a regiones del contorno del círculo
def asignar_valores(event, x, y, flags, param):
    global circulo_puntos, valores_regiones, dibujando_circulo, regiones_dibujadas
    
    if dibujando_circulo and event == cv2.EVENT_LBUTTONDOWN:
        circulo_puntos.append((x, y))
        if len(circulo_puntos) == 2:
            dibujando_circulo = False
            centro = circulo_puntos[0]
            radio = int(np.sqrt((circulo_puntos[1][0] - centro[0])**2 + (circulo_puntos[1][1] - centro[1])**2))
            cv2.circle(recorte, centro, radio, (255, 0, 0), 2)
            cv2.imshow("Region de Interes con Circulo", recorte)
            print("Haga clic en diferentes puntos del contorno del círculo para asignar valores.")
            regiones_dibujadas = True
    
    elif regiones_dibujadas and event == cv2.EVENT_LBUTTONDOWN:
        if circulo_puntos:
            centro = circulo_puntos[0]
            radio = int(np.sqrt((circulo_puntos[1][0] - centro[0])**2 + (circulo_puntos[1][1] - centro[1])**2))
            distancia = np.sqrt((x - centro[0])**2 + (y - centro[1])**2)
            
            if abs(distancia - radio) < 10:  # Permitir un margen de error para el clic en el contorno
                valor = input("Ingrese el valor para esta región: ")
                angulo = math.degrees(math.atan2(y - centro[1], x - centro[0]))
                if angulo < 0:
                    angulo += 360
                valores_regiones.append((angulo, valor))
                valores_regiones.sort()  # Ordenar los valores por ángulo
                cv2.circle(recorte, (x, y), 5, (0, 255, 0), -1)  # Marca la región seleccionada
                cv2.putText(recorte, valor, (x + 10, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
                cv2.imshow("Region de Interes con Circulo", recorte)
    
    elif event == cv2.EVENT_RBUTTONDOWN:  # Guardar valores en un archivo Excel
        df = pd.DataFrame(valores_regiones, columns=['Angulo', 'Valor'])
        df.to_excel('valores_manometro.xlsx', index=False)
        print("Valores guardados en 'valores_manometro.xlsx'.")

# Cargar el video y leer el primer frame
cap = cv2.VideoCapture('hiv00003.mp4')
ret, frame = cap.read()
cap.release()

if ret:
    recorte = frame.copy()
    
    # Mostrar la imagen y configurar el callback del mouse
    cv2.imshow("Region de Interes", recorte)
    r = cv2.selectROI("Region de Interes", recorte, fromCenter=False, showCrosshair=True)
    recorte = recorte[int(r[1]):int(r[1]+r[3]), int(r[0]):int(r[0]+r[2])]
    cv2.imwrite('tu_imagen.jpg', recorte)  # Guardar la imagen recortada
    cv2.destroyAllWindows()
    
    cv2.imshow("Region de Interes con Circulo", recorte)
    cv2.setMouseCallback("Region de Interes con Circulo", asignar_valores)
    
    # Permitir al usuario dibujar el círculo
    print("Haga clic en el centro del círculo y luego en un punto en su borde.")
    dibujando_circulo = True
    cv2.waitKey(0)
    
    print("Haga clic en diferentes puntos del contorno del círculo para asignar valores.")
    cv2.waitKey(0)
    
    print("Haga clic con el botón derecho para guardar los valores en un archivo Excel.")
    cv2.waitKey(0)
    
    cv2.destroyAllWindows()
else:
    print("No se pudo leer el primer frame del video.")


Haga clic en el centro del círculo y luego en un punto en su borde.
Haga clic en diferentes puntos del contorno del círculo para asignar valores.


Ingrese el valor para esta región:  -1
Ingrese el valor para esta región:  0
Ingrese el valor para esta región:  1
Ingrese el valor para esta región:  2
Ingrese el valor para esta región:  3
Ingrese el valor para esta región:  4
Ingrese el valor para esta región:  4.5


Haga clic en diferentes puntos del contorno del círculo para asignar valores.
Haga clic con el botón derecho para guardar los valores en un archivo Excel.
Valores guardados en 'valores_manometro.xlsx'.


In [32]:
import cv2
import pandas as pd
import numpy as np
import math

# Función para determinar el valor según el ángulo con interpolación
def determinar_valor_interpolado(angulo, valores_regiones):
    if valores_regiones:
        valores_regiones = sorted(valores_regiones)  # Asegurar que los valores estén ordenados por ángulo
        for i in range(len(valores_regiones)):
            ang1, val1 = valores_regiones[i]
            ang2, val2 = valores_regiones[(i + 1) % len(valores_regiones)]  # Cierre del círculo
            if ang1 <= angulo <= ang2 or (ang2 < ang1 and (angulo >= ang1 or angulo <= ang2)):  # Manejar cierre del círculo
                # Interpolación lineal
                if ang2 < ang1:
                    ang2 += 360  # Manejar el cruce de 360 grados
                angulo_rel = angulo if angulo >= ang1 else angulo + 360  # Ajustar ángulo relativo
                interpolacion = (angulo_rel - ang1) / (ang2 - ang1)
                val = float(val1) + interpolacion * (float(val2) - float(val1))
                return val
        return float(valores_regiones[0][1])  # Si el ángulo es mayor a todos, devolver el primer valor
    return None

# Función para manejar el clic del mouse y mostrar el valor correspondiente
def mostrar_valor(event, x, y, flags, param):
    global circulo_puntos, valores_regiones, imagen_recorte, resultados, segundos_actuales
    
    if event == cv2.EVENT_LBUTTONDOWN:
        if circulo_puntos:
            centro = circulo_puntos[0]
            radio = int(np.sqrt((circulo_puntos[1][0] - centro[0])**2 + (circulo_puntos[1][1] - centro[1])**2))
            distancia = np.sqrt((x - centro[0])**2 + (y - centro[1])**2)
            
            if abs(distancia - radio) < 10:  # Permitir un margen de error para el clic en el contorno
                angulo = math.degrees(math.atan2(y - centro[1], x - centro[0]))
                if angulo < 0:
                    angulo += 360
                
                valor = determinar_valor_interpolado(angulo, valores_regiones)
                if valor is not None:
                    print(f"El valor en el ángulo {angulo:.2f} es {valor:.2f}.")
                    resultados.append({'Segundo': segundos_actuales, 'Valor': valor})
                    cv2.putText(imagen_recorte, f"{valor:.2f}", (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)
                    cv2.imshow("Recorte de Frame con Circulo", imagen_recorte)
                else:
                    print("No se encontró un valor correspondiente.")

# Cargar valores de referencia desde el archivo Excel
df = pd.read_excel('valores_manometro.xlsx')
valores_regiones = df.values.tolist()

# Inicializar archivo Excel para resultados
resultados = []

# Cargar el video
video_path = 'hiv00014.mp4'
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
intervalo = 15  # Intervalo de 15 segundos
frame_id = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    segundos_actuales = frame_id / fps
    if segundos_actuales % intervalo == 0:
        # Recortar la región de interés del frame
        x, y, w, h = r  # Ajusta las coordenadas y el tamaño según tu necesidad
        imagen_recorte = frame[y:y+h, x:x+w]
        
        cv2.imshow("Recorte de Frame con Circulo", imagen_recorte)
        cv2.setMouseCallback("Recorte de Frame con Circulo", mostrar_valor)
        
        # Esperar hasta que el usuario haga clic
        print(f"Imagen capturada en {segundos_actuales:.2f} segundos.")
        cv2.waitKey(0)
        
        cv2.destroyAllWindows()
    
    frame_id += 1

cap.release()

# Guardar resultados en un archivo Excel
df_resultados = pd.DataFrame(resultados)
df_resultados.to_excel('resultados_video.xlsx', index=False)
print("Resultados guardados en 'resultados_video.xlsx'.")


Imagen capturada en 0.00 segundos.
El valor en el ángulo 182.42 es -0.33.
Imagen capturada en 15.00 segundos.
El valor en el ángulo 183.97 es -0.29.
Imagen capturada en 30.00 segundos.
El valor en el ángulo 183.97 es -0.29.
Imagen capturada en 45.00 segundos.
El valor en el ángulo 183.97 es -0.29.
Imagen capturada en 60.00 segundos.
El valor en el ángulo 183.97 es -0.29.
Imagen capturada en 75.00 segundos.
El valor en el ángulo 183.97 es -0.29.
Imagen capturada en 90.00 segundos.
El valor en el ángulo 183.09 es -0.31.
Imagen capturada en 105.00 segundos.
El valor en el ángulo 183.09 es -0.31.
El valor en el ángulo 193.11 es -0.07.
Imagen capturada en 120.00 segundos.
El valor en el ángulo 193.28 es -0.06.
Imagen capturada en 135.00 segundos.
El valor en el ángulo 193.28 es -0.06.
Imagen capturada en 150.00 segundos.
El valor en el ángulo 193.28 es -0.06.
Imagen capturada en 165.00 segundos.
El valor en el ángulo 193.28 es -0.06.
Imagen capturada en 180.00 segundos.
El valor en el ángul

In [ ]:
import cv2

# Variables globales
drawing = False  # Verdadero cuando el mouse está presionado
ix, iy = -1, -1  # Coordenadas iniciales
fx, fy = -1, -1  # Coordenadas finales

# Función para dibujar el rectángulo
def dibujar_rectangulo(event, x, y, flags, param):
    global ix, iy, fx, fy, drawing

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            fx, fy = x, y
            img_temp = frame.copy()
            cv2.rectangle(img_temp, (ix, iy), (fx, fy), (0, 255, 0), 2)
            cv2.imshow('Frame', img_temp)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        fx, fy = x, y
        cv2.rectangle(frame, (ix, iy), (fx, fy), (0, 255, 0), 2)
        cv2.imshow('Frame', frame)

# Cargar el video
video_path = 'hiv00010.mp4'
cap = cv2.VideoCapture(video_path)

# Verificar si el video se abrió correctamente
if not cap.isOpened():
    print("Error al abrir el video.")
    exit()

# Leer el primer frame
ret, frame = cap.read()
if not ret:
    print("No se pudo leer el primer frame.")
    cap.release()
    exit()

# Crear una ventana y configurar la función del mouse
cv2.namedWindow('Frame')
cv2.setMouseCallback('Frame', dibujar_rectangulo)

# Mostrar el frame y permitir la selección del ROI
print("Dibuja un rectángulo alrededor de la región que deseas recortar y presiona cualquier tecla para continuar.")
cv2.imshow('Frame', frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Verificar que las coordenadas finales sean válidas
if ix < fx and iy < fy:
    # Recortar la región de interés
    recorte = frame[iy:fy, ix:fx]

    # Guardar la región recortada como una imagen
    output_image_path = 'recorte_frame.jpg'
    cv2.imwrite(output_image_path, recorte)

    # Mostrar la imagen recortada
    cv2.imshow("Recorte de Frame", recorte)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    print(f"Imagen recortada guardada en {output_image_path}.")
else:
    print("Las coordenadas de recorte no son válidas.")

# Liberar recursos
cap.release()
